In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib as mpl
from scipy.signal import savgol_filter
import yaml
import time
from pathlib import Path
plt.rcParams["animation.html"] = "jshtml"

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy.signal import find_peaks
import tardigrade_functions as tg

In [ ]:
def angle(row):
    v1 = [row.X1, row.Y1]
    v2 = [row.X2, row.Y2]
    return np.arccos(np.dot(v1, v2)/np.linalg.norm(v1)/np.linalg.norm(v2))

def vector_angle(XY1, XY2, diff_periods=3):
    cms_v1 = XY1.diff(diff_periods).values
    cms_v2 = XY2.diff(diff_periods).values
    cms_v = pd.DataFrame([*cms_v1.T,*cms_v2.T], index=['X1','Y1','X2','Y2']).T
    return cms_v.apply(lambda row: angle(row), axis=1)

def safe_round(a, decimals=0, axis=-1):
    """
    round safely to decimals. I.e. Keep the sum axross the axis constant. Add deviations to the fairest index
    Args:
        a (array)
        decimals (int) defaulft 0
        axis (int) default -1
    Returns:
        a_r (array) rounded array
    """
    sum_ = np.sum(a, axis=axis)
    a_r = np.round(a, decimals=decimals)
    sum_r = np.sum(a_r, axis=axis)
    devi = np.round(sum_r - sum_, decimals=decimals)
    sortidx_tochange = [np.arange(np.min([0,d]), np.max([0,d])).astype(int) for d in devi * 10**decimals]
    sortidx = np.argsort(a-a_r)#[devi]
    for i, (s,c,d) in enumerate(zip(sortidx, sortidx_tochange, devi)):
        if c.size > 0:
            for j in c:
                a_r[i][np.array(s)[j]] -= d/len(c)
    return a_r

In [ ]:
date = time.strftime("%Y%m%d")
home = os.path.expanduser("~")
out_path = os.path.join(home, f'data/LB016/out_{date}')
pose_path = os.path.join(home, 'data/LB016/poseestimation')
stage_path = os.path.join(home, 'data/LB016/stage/')

stage_files = {}
pose_files = {}
for fn in os.listdir(stage_path):
    if not fn.startswith('.') and fn.endswith('txt'):
        id = fn.split('-')[0]
        stage_files[id] = os.path.join(stage_path, fn)

fpath = Path(pose_path)
for id in stage_files.keys():
    for fn in list(filter(Path.is_file, fpath.glob(id+'*.csv'))):
        if id in pose_files.keys():
            print(f'Warning {id} already in pose_files')
            break
        pose_files[id] = os.path.join(pose_path, fn)


In [ ]:
config_path = "config.yaml"
config = yaml.safe_load(open(config_path, "r"))

fps = config['video']['fps']

bodyparts = config['analysis']['bodyparts']
limbs = config['analysis']['limbs']
limbs_ord = config['analysis']['limbs_ord']
pairs = config['analysis']['pairs']
bp_pairs = config['analysis']['bp_pairs']

bp_color_dict = config['color']['bp_color']
roi_defined = config['analysis']['roi_defined']

idx = pd.IndexSlice

In [ ]:
for i,id in enumerate(pose_files):
    print(id)
    stage_fn = stage_files[id]
    DLCfile_org = pd.read_csv(pose_files[id], header=[0,1,2])
    outfn = os.path.join(out_path,id)
    if not os.path.isdir(outfn):
        os.makedirs(outfn)

    ### preprocessing
    # set values lower 50% likelihood to nan
    mask = DLCfile_org.xs('likelihood', level=2, axis=1).apply(lambda s: s<=.7)
    maskDLC = DLCfile_org.droplevel(2, axis=1)
    maskDLC = maskDLC.align(mask,'left', axis=1)[1].fillna(False)
    
    old_idx = maskDLC.columns.to_frame() # convert to DataFrame
    old_idx.insert(2, 2, DLCfile_org.columns.get_level_values(2)) # add new level
    maskDLC.columns = pd.MultiIndex.from_frame(old_idx)
    maskDLC.loc[:,idx[:,:,'likelihood']] = False
    DLCfile_org[maskDLC.values] = np.nan
    
    # interpolate and make copy
    DLCfile = DLCfile_org.copy()
    DLCfile = DLCfile.interpolate(limit_direction='both')
    # load stage and reorientate coordinates to stage 
    DLC, DLCorg, stage, centerline, matrix = tg.reformat_coordinatefile(stage_fn, DLCfile, bodyparts, pairs)

    stage_Xrange = np.subtract(np.max(stage['X']), np.min(stage['X']))
    stage_Yrange = np.subtract(np.max(stage['Y']), np.min(stage['Y']))
    DLC_Xrange = np.subtract(np.max(DLC.loc[:,idx[:,:,'x']]), np.min(DLC.loc[:,idx[:,:,'x']]))
    DLC_Yrange = np.subtract(np.max(DLC.loc[:,idx[:,:,'x']]), np.min(DLC.loc[:,idx[:,:,'x']]))

    plt.plot(DLCorg.loc[:500, idx[:,'leg_2_left','x']],DLCorg.loc[:500, idx[:,'leg_2_left','y']])
    plt.plot(DLC.loc[:500, idx[:,'leg_2_left','x']],DLC.loc[:500, idx[:,'leg_2_left','y']])


    ### Tracks

    # CMS Tracks
    fig, ax = plt.subplots(figsize=(stage_Xrange, stage_Yrange))
    scale = 1000
    stage = stage.rolling(10, min_periods=1).mean()
    ax.plot(stage.loc[:,'Xstage'],stage.loc[:,'Ystage'], label='stage')
    ax.plot(stage.loc[:,'Xcms'],stage.loc[:,'Ycms'], label='aligned image')

    scalebar = AnchoredSizeBar(ax.transData,
                                scale, f'{scale} um', 'lower right', 
                                pad=1,
                                color='black',
                                frameon=False,
                                size_vertical=10,)
    ax.add_artist(scalebar)
    plt.axis('equal')
    plt.axis('off')
    plt.savefig(os.path.join(outfn, f'{id}_tracks.pdf'))
    plt.show()


    # Bodypart Tracks
    fig, ax = plt.subplots(figsize=(DLC_Xrange/500, DLC_Yrange/500))
    for bp in bodyparts:
        ax.plot(DLC.loc[:,idx[:,bp,'x']],
                DLC.loc[:,idx[:, bp,'y']], label = bp, color=bp_color_dict[bp], 
                )
    plt.legend(bbox_to_anchor=(1,1.02), loc='upper left')
    scale = 1000
    scalebar = AnchoredSizeBar(ax.transData,
                                scale, f'{scale} um', 'lower left', 
                                pad=-0.5,
                                color='black',
                                frameon=False,
                                size_vertical=10,)
    ax.add_artist(scalebar)
    plt.axis('equal')
    plt.axis('off')
    plt.savefig(os.path.join(outfn, f'{id}_tracks_limbs.pdf'), bbox_inches='tight')
    plt.show()
    
    # BodyPart Track Zoomed IN
    fig, ax = plt.subplots()
    for bp in bodyparts:
        ax.plot(DLC.loc[:20*fps,idx[:,bp,'x']],
                DLC.loc[:20*fps,idx[:, bp,'y']], label = bp, color=bp_color_dict[bp], 
                )
    plt.legend(bbox_to_anchor=(1,1.02), loc='upper left')
    scale = 300
    scalebar = AnchoredSizeBar(ax.transData,
                                scale, f'{scale} um', 'lower left', 
                                pad=-0.5,
                                color='black',
                                frameon=False,
                                size_vertical=2,)
    ax.add_artist(scalebar)
    plt.axis('equal')
    plt.axis('off')
    plt.savefig(os.path.join(outfn, f'{id}_tracks_limbs_first20s.pdf'), bbox_inches='tight')
    plt.show()

    ### DLC

    # likelood and and x y position
    fig, axs = plt.subplots(2,2, figsize=(10,5))
    axs = axs.flatten()
    for i, bp in enumerate(bodyparts):
        axs[0].plot(DLCfile.loc[:,idx[:,bp,'likelihood']], color=bp_color_dict[bp], alpha=.5)
        axs[2].plot(DLCfile.loc[:,idx[:,bp,'x']], color=bp_color_dict[bp], alpha=.5)
        axs[3].plot(DLCfile.loc[:,idx[:,bp,'y']], color=bp_color_dict[bp], alpha=.5, label=bp)
    axs[1].plot(DLCfile.loc[:,idx[:,bp,'likelihood']].mean(axis=1))
    for ax in axs[:2]:
        ax.set_ylim(-.1,1.1)
    axs[0].set_ylabel('likelihood')
    axs[1].set_ylabel('likelihood mean')
    axs[2].set_ylabel('x position')
    axs[3].set_ylabel('y position')
    plt.legend(loc='upper left', bbox_to_anchor=(1,2.25))
    plt.savefig(os.path.join(outfn, f'{id}_likelihood_position.pdf'), bbox_inches='tight')
    plt.show()

    ### Analysis

    # calculate the velocity from aligned cms data
    stage['velocity'] = np.sqrt(stage['Xcms'].diff(periods=3)**2 + stage['Ycms'].diff(periods=3)**2)/3*fps
    # calculate the velocity for each bodypart from aligned data
    DLCvelo = np.sqrt((DLC.loc[:,idx[:,:,'x']].diff(periods=3)**2).droplevel(2, axis=1) + (DLC.loc[:,idx[:,:,'y']].diff(periods=3)**2).droplevel(2, axis=1))/3*fps
    DLCvelo = DLCvelo.droplevel(0, axis=1)
    DLCvelo = DLCvelo.fillna(0)
    _ = savgol_filter(DLCvelo, int(fps//3), 2, axis=0)
    DLCvelo_smooth = pd.DataFrame(_, columns=DLCvelo.columns, index=DLCvelo.index)

    # calculate angle between bp and gcms movement vector
    track_angle = vector_angle(stage[['Xcms','Ycms']].iloc[15:],stage[['Xcms','Ycms']].iloc[:-15], 1)
    moveangle = pd.DataFrame([]).reindex_like(DLCvelo)
    for i,bp in enumerate(bodyparts):
        moveangle[bp] = vector_angle(stage[['Xcms','Ycms']], DLC.loc[:,idx[:, bp,['x','y']]], 2)
    alignedmovement = abs(moveangle/np.pi - 1)# -.5) * 2

    # compute straightend tardigrade
    cx,cy = centerline.loc[:,idx[:,'x']].droplevel(1, axis=1), centerline.loc[:,idx[:,'y']].droplevel(1, axis=1)
    shiftrear_x, shiftrear_y = (cx['rear']-cx['leg_3'])*.3, (cy['rear']-cy['leg_3'])*.3
    cx['rear'] = cx['rear']+shiftrear_x
    cy['rear'] = cy['rear']+shiftrear_y
    cx,cy = cx.values,cy.values
    cdist = np.sqrt(np.diff(cx, axis=1)**2 + np.diff(cy, axis=1)**2)
    cdist_norm = (cdist / np.sum(cdist, axis=1)[:,np.newaxis] * 100)
    cdist_cum= np.cumsum(safe_round(cdist_norm, decimals=0, axis=-1), axis=1).astype(int)
    CLx = np.array([np.interp(np.arange(100), [0]+list(cdist_cum[i]), cx[i]) for i in range(len(cx))])
    CLy = np.array([np.interp(np.arange(100), [0]+list(cdist_cum[i]), cy[i]) for i in range(len(cy))])
    CL = np.stack([CLx,CLy])
    # extract nearest cl point for each bp and compute distance to that
    CLbp = pd.DataFrame([]).reindex_like(DLC)
    for bp in bodyparts:
        m = 1 if 'right' in bp else -1
        cl_bp_diff = CL.T - DLC.loc[:,idx[:,bp,['x','y']]].values=
        CLbp_idx = np.linalg.norm(cl_bp_diff, axis=2).argmin(axis=0)
        DLbp_dist = np.linalg.norm(CL[:,np.arange(len(CLbp_idx)),CLbp_idx].T - DLC.loc[:,idx[:,bp,['x','y']]], axis=1)
        CLbp.loc[:,idx[:,bp,['x','y']]] = np.stack([abs(CLbp_idx-100), DLbp_dist*m/np.sum(cdist, axis=1)*100]).T
    
    # detect swing bouts by peak detection on bodypart velocity
    swing_bouts = pd.DataFrame([])
    bp_peaks = {}
    CLbp_norm = (CLbp - CLbp.mean(axis=0)) / CLbp.std(axis=0)
    CLbp_x = CLbp_norm.loc[:,idx[:,:,'x']].droplevel(level=[0,2],axis=1).fillna(0)
    CLbp_bg = CLbp_x.rolling(fps, min_periods=1, center=True).mean()

    CLbp_x_smooth = pd.DataFrame(savgol_filter((CLbp_x-CLbp_bg).T, 15, 2).T)
    CLbp_x_smooth.columns = CLbp_x.columns

    dCLbp_x_smooth = pd.DataFrame(savgol_filter(CLbp_x_smooth.T, 10, 2, deriv=1).T)
    dCLbp_x_smooth.columns = CLbp_x.columns

    ddCLbp_x_smooth = pd.DataFrame(savgol_filter(CLbp_x_smooth.T, 10, 2, deriv=2).T)
    ddCLbp_x_smooth.columns = CLbp_x.columns

    bp_peaks = {}
    bp_troughs = {}
    for bp in limbs:
        troughs, _ = find_peaks(dCLbp_x_smooth[bp], distance=1, height=0)
        peaks, peak_props = find_peaks(-ddCLbp_x_smooth[bp], distance=1,  height=0)
        tp_pairs = np.vstack([np.array([troughs[np.argmax(1/(p-troughs))], p]) for p in peaks])
        uni,cnt = np.unique(tp_pairs[:,0], return_counts=True)
        uni_i = [np.where(tp_pairs==u)[0] for u in uni[cnt>1]]
        try:
            drop_i = np.concatenate([i[1:] for i in uni_i]).flatten() # select closest trough
        except:
            drop_i = []
        dropt = np.full(len(tp_pairs), True)
        dropt[drop_i] = False
        tp_pairs = tp_pairs[dropt]
        
        peaks_bout = tg.ethogram_fromOnOff(tp_pairs[:,0], tp_pairs[:,1], arr_len=len(CLbp_x))
        
        swing_bouts[bp] =  peaks_bout.astype(bool)
        bp_troughs[bp] =  tp_pairs[:,0]
        bp_peaks[bp] =  tp_pairs[:,1]

    # find turn points
    stage[['Xcms_smooth','Ycms_smooth']] = stage[['Xcms','Ycms']].rolling(fps*2, center=True, min_periods=1).mean()
    track_angle = vector_angle(stage[['Xcms_smooth','Ycms_smooth']].iloc[fps:],stage[['Xcms_smooth','Ycms_smooth']].iloc[:-fps], fps)
    turns, dirchange_props = find_peaks(track_angle, distance=fps*5, height=np.pi/4)
    fig = plt.figure(figsize=(5,5))

    # find straight runs
    cms_savgol = savgol_filter(stage[['Xcms','Ycms']],fps*10,1, axis=0) # filter track with savgol to remove smaller bends
    macro_angle = vector_angle(pd.DataFrame(cms_savgol[fps:]),pd.DataFrame(cms_savgol[:-fps]), 15) # calculate angle
    straight = macro_angle < .03 # select bouts underneeth threshold
    straight = straight.astype(int) # prep interpolation
    straight[straight==0] = np.nan # prep interpolation
    straight = straight.interpolate(limit_direction='both', limit=fps) # interpolate within 1 sec
    straight = straight.fillna(0).astype(bool)
    macro_angle[straight] = 0

    ### Plots

    # Bodyparts first frame and straightend
    fig, axs = plt.subplots(1,2, figsize=(6,3))
    for bp in bodyparts:
        axs[0].scatter(DLC.loc[0,idx[:, bp,'x']],
                DLC.loc[0,idx[:, bp,'y']], label=bp, color=bp_color_dict[bp])
        axs[1].scatter(CLbp.loc[0,idx[:,bp,'x']],
                       CLbp.loc[0,idx[:,bp,'y']], label=bp, color=bp_color_dict[bp])
        
    for p in pairs:
        axs[0].scatter(centerline.loc[0,idx[p,'x']],
                centerline.loc[0,idx[p,'y']], label = p, marker='x', color='k')
    for ax in axs:
        ax.axis('equal')
    axs[0].axis('off')
    axs[1].set_xticks([])
    axs[1].set_yticks([])
        
    scale = 100
    scalebar = AnchoredSizeBar(axs[0].transData,
                                scale, f'{scale} um', 'lower left', 
                                pad=-0.5,
                                color='black',
                                frameon=False,
                                size_vertical=1,)
    axs[0].add_artist(scalebar)

    axs[1].legend(bbox_to_anchor=(1,1), loc='upper left')
    plt.savefig(os.path.join(outfn, f'{id}_frame0.pdf'), bbox_inches='tight')
    plt.show()

    
    # velocity of all legs
    fig, axs = plt.subplots(2, 1, figsize=(15,5), sharex=True)
    for p in bp_pairs:
        axs[0].plot(CLbp_x_smooth[f'{p}_left'], label=f'{p}_left', c=bp_color_dict[f'{p}_left'])
        axs[0].scatter(bp_peaks[f'{p}_left'], CLbp_x_smooth[f'{p}_left'].iloc[bp_peaks[f'{p}_left']], c=bp_color_dict[f'{p}_left'])
        axs[0].scatter(bp_troughs[f'{p}_left'], CLbp_x_smooth[f'{p}_left'].iloc[bp_troughs[f'{p}_left']], c=bp_color_dict[f'{p}_left'], marker='x')
        axs[1].plot(CLbp_x_smooth[f'{p}_right'], label=f'{p}_right', c=bp_color_dict[f'{p}_right'])
        axs[1].scatter(bp_peaks[f'{p}_right'], CLbp_x_smooth[f'{p}_right'].iloc[bp_peaks[f'{p}_right']], c=bp_color_dict[f'{p}_right'])
        axs[1].scatter(bp_troughs[f'{p}_right'], CLbp_x_smooth[f'{p}_right'].iloc[bp_troughs[f'{p}_right']], c=bp_color_dict[f'{p}_right'], marker='x')
    for ax in axs:
        #ax.set_ylim(0,400)
        ax.legend(bbox_to_anchor=(1,1), loc='upper left')
        ax.set_xlim(0*fps,20*fps)
        ax.set_xticks(np.arange(*ax.get_xlim(),fps))
        ax.set_xticklabels(np.arange(*np.divide(ax.get_xlim(),fps).astype(int),1))
    plt.savefig(os.path.join(outfn, f'{id}_legvelo.png'), bbox_inches='tight')
    plt.show()

    # example plot of hindleg for swing detection
    leg = 'hindleg_left'
    fig = plt.figure(figsize=(20,5))
    plt.plot(CLbp_x_smooth[leg][:20*fps])
    plt.plot((swing_bouts[leg])[:20*fps])
    plt.xticks(np.arange(0, 20*fps, fps), (np.arange(0, 20*fps, fps)/fps).astype(int));
    plt.savefig(os.path.join(outfn, f'{id}_{leg}_peak2swing.png'), bbox_inches='tight')
    plt.show()

    # plot turns and straight runs
    plt.plot(*stage[['Xcms_smooth','Ycms_smooth']].T.values)
    plt.scatter(*stage[['Xcms_smooth','Ycms_smooth']].iloc[turns].T.values)
    plt.scatter(*stage[['Xcms_smooth','Ycms_smooth']][:-15][straight].T.values, c='pink')
    plt.axis('equal')
    plt.axis('off')
    plt.show()

    # get selectable index for straight bouts, remove index within 5 sec to straight bout end, to ensure straight bout visualisation
    straight_idx = straight[straight].index # get index
    straight_end = np.append(straight_idx[straight_idx.diff(-1) < -1], straight_idx[-1]) # get index for bout end
    straight_end5s = (straight_end.repeat(5*fps).reshape(len(straight_end), 5*fps) - np.arange(5*fps)).flatten() #extend bout end 5 sec backwards
    straight_start = [s for s in straight_idx if s not in straight_end5s] # filter index
    roi_straight = np.random.choice(straight_start, 5) # select

    roi_turns = np.random.choice(turns-3*fps, 5, replace=False)
        
    CLbp_forimg = ((CLbp_x_smooth[limbs_ord]-np.min(CLbp_x_smooth[limbs_ord],axis=0))/(np.max(CLbp_x_smooth[limbs_ord],axis=0) - np.min(CLbp_x_smooth[limbs_ord],axis=0)))+[-0.5,.5,1.5,2.5,3.5,4.5,5.5,6.5]
    for k,roi in {'straight':roi_straight, 'turns':roi_turns}.items():
        fig, axs = plt.subplots(len(roi),2, figsize=(7,10),)
        axs = axs if isinstance(axs, np.ndarray) else np.array([axs])
        for i, rng in enumerate(roi):
            rng = (rng, rng+6*fps)
            #print(rng)
            axs[i,0].imshow(swing_bouts[limbs_ord].astype(int).T,interpolation='none',rasterized=True, cmap='Greys_r', origin='lower')
            for bp in limbs:
                axs[i,0].plot(CLbp_forimg[bp].values, c=bp_color_dict[bp], alpha=.5, lw=1.5)
            axs[i,0].set_yticks(range(len(limbs)))
            axs[i,0].set_yticklabels(limbs_ord)
            axs[i,0].axis('auto')
            axs[i,0].set_xlim(*rng)
            axs[i,0].set_xticks(np.arange(*rng,fps))
            axs[i,0].set_xticks(np.arange(*rng,5), minor=True)
            axs[i,0].set_xticklabels(np.arange(*rng,fps)//fps)

            for bp in bodyparts:
                axs[i,1].plot(DLC.loc[rng[0]:rng[1],idx[:,bp,'x']],
                        DLC.loc[rng[0]:rng[1],idx[:, bp,'y']], color=bp_color_dict[bp], 
                        )
            scale = 10
            scalebar = AnchoredSizeBar(axs[i,1].transData,
                                        scale, f'{scale} um', 'lower left', 
                                        pad=-0.5,
                                        color='black',
                                        frameon=False,
                                        size_vertical=2,)
            axs[i,1].add_artist(scalebar)
            axs[i,1].axis('equal')
            axs[i,1].axis('off')

        plt.savefig(os.path.join(outfn, f'{id}_hildebrand_{k}example.pdf'), bbox_inches='tight')
        plt.show()
    
    # save
    DLCvelo.to_csv(os.path.join(outfn, f'{id}_DLCvelo.csv'))
    DLC.to_csv(os.path.join(outfn, f'{id}_DLC-coorinstage.csv'))
    swing_bouts.to_csv(os.path.join(outfn, f'{id}_swing.csv'))
    stage.to_csv(os.path.join(outfn, f'{id}_stage.csv'))
    track_angle.to_csv(os.path.join(outfn, f'{id}_trackangle.csv'))
    pd.DataFrame(turns, columns=['turn frames']).to_csv(os.path.join(outfn, f'{id}_turns.csv'))
    straight.to_csv(os.path.join(outfn, f'{id}_straight.csv'))
    CLbp.to_csv(os.path.join(outfn, f'{id}_CLbp.csv'))